In [ ]:
# Copyright 2026 The AI Edge Quantizer Authors.
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.
# ==============================================================================

<table class="tfo-notebook-buttons" align="left">
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/google-ai-edge/LiteRT-LM/blob/main/python/colabs/Getting Started with LiteRT-LM Python API.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
  </td>
  <td>
    <a target="_blank" href="https://github.com/google-ai-edge/ai-edge-quantizer/blob/main/python/colabs/Getting Started with LiteRT-LM Python API.ipynb"><img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />View source on GitHub</a>
  </td>
</table>

# Getting Started with LiteRT-LM Python API

This notebook walks through the [LiteRT-LM Python API](https://ai.google.dev/edge/litert-lm/python) with [Gemma 4 E2B](https://huggingface.co/litert-community/gemma-4-E2B-it-litert-lm).

## 1. Installation

First, let's install the LiteRT-LM API from PyPI. We will need the Hugging Face package to download the model.

In [ ]:
!pip install litert-lm huggingface_hub

## 2. Download the [Gemma 4 E2B model](https://huggingface.co/litert-community/gemma-4-E2B-it-litert-lm)

In [ ]:
from huggingface_hub import hf_hub_download

# Download the model file using your authenticated session
model_path = hf_hub_download(
    repo_id="litert-community/gemma-4-E2B-it-litert-lm",
    filename="gemma-4-E2B-it.litertlm",
    local_dir=".",
    local_dir_use_symlinks=False
)

# Rename to the expected filename if necessary
import os
if os.path.exists("gemma-4-E2B-it.litertlm"):
    os.rename("gemma-4-E2B-it.litertlm", "model.litertlm")
print("Model downloaded successfully as model.litertlm")

## 3. Sending Messages (Synchronous)

You can send a simple string or a full message structure synchronously.

Note that the first initialization of `model.litertlm` will take longer. Future initialization will be faster with the cache.

In [ ]:
import litert_lm

with litert_lm.Engine("model.litertlm") as engine:
  with engine.create_conversation() as conversation:
    response = conversation.send_message("What is the capital of France?")
    print(f"Response: {response['content'][0]['text']}")

Response: The capital of France is **Paris**.


## 4. Sending Messages (Asynchronous/Streaming)

Streaming the response as it's generated.

In [ ]:
import litert_lm

with litert_lm.Engine("model.litertlm") as engine:
  with engine.create_conversation() as conversation:
    for chunk in conversation.send_message_async("Tell me a joke."):
      print(chunk["content"][0]["text"], end="", flush=True)

Here's one for you:

Why don't scientists trust atoms?

**Because they make up everything!**

## 5. Multi-Modality

Gemma 3n supports multi-modality. Here's how to interact with it using audio or images (if supported).

In [ ]:
# Download the audio file for this example
!wget https://github.com/google-ai-edge/LiteRT-LM/raw/refs/heads/main/runtime/testdata/have_a_wonderful_day.wav -O have_a_wonderful_day.wav

In [ ]:
import litert_lm

user_message = {
    "role": "user",
    "content": [
        {"type": "audio", "path": "/content/have_a_wonderful_day.wav"},
        {"type": "text", "text": "Describe this audio. What does it say?"},
    ],
}

# Initialize the engine with Audio backend.
with litert_lm.Engine("model.litertlm", audio_backend=litert_lm.Backend.CPU) as engine:
  with engine.create_conversation() as conversation:
    for chunk in conversation.send_message_async(user_message):
      print(chunk["content"][0]["text"], end="", flush=True)

The audio says, "Have a wonderful day."


## 6. Defining and Using Tools

You can define Python functions as tools that the model can call automatically. Note that this requires models with tool support.

In [ ]:
import litert_lm

def product(numbers: float) -> float:
    """Compute the product of the given numbers.

    Args:
      numbers: The list of numbers.
    """
    print("[tool call] 'product' with numbers", numbers)

    result = 1
    for x in numbers:
        result *= x
    return result

with litert_lm.Engine("model.litertlm") as engine:
  with engine.create_conversation(tools=[product]) as conversation:
    # The model will call add_numbers automatically if it needs to sum values
    for chunk in conversation.send_message_async("What is the product of 12.34 and 98.76?"):
      print(chunk["content"][0]["text"], end="", flush=True)

[tool call] 'product' with numbers [12.34, 98.76]
The product of 12.34 and 98.76 is 1218.6984.